# Chapter 2 — Computing with vectors and matrices · Notebook to complete

*Maths for AI, Step by Step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch02_a_completer.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

The cells to edit are marked **TO COMPLETE**. Run the cells in order: as long as an answer is missing, the checks say so without an error. The solution is in `ch02_corrige.ipynb`, on the same companion page.

**Prediction rule of the chapter** (given, not fitted): weights $\mathbf{w} = (3, -8)$ in k€/m² and k€/km, intercept $b = 45$ k€.

With the districts (lesson 2.4), the rule becomes $\mathbf{w} = (3, -8, 12, -10)$, $b = 45$ k€, with Gare (the station) as the reference district.

**Contents.** Data · The lesson computations · Exercises 2.1 to 2.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colors of the book: one color = one role (observed, predicted, deviation, colleague's line)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Data

The identifiers (D2.1, D2.E2) are those of the book and of the CSV files.

In [ ]:
# Chapter 2 data, copied from the book's CSV files (do not modify)

# D2.1 — the seven homes with their distance to the center (lessons 2.1 to 2.3)
homes = pd.DataFrame({
    'home':    ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    'floor_area':  [32, 45, 54, 70, 85, 98, 62],
    'distance': [3.0, 1.5, 2.5, 1.0, 2.5, 0.5, 2.5],
    'price':   [118, 172, 185, 245, 279, 335, 209],
})

# D2.4 — five homes in the east of the city, with their district (lesson 2.4)
east = pd.DataFrame({
    'home':    ['K', 'L', 'M', 'N', 'O'],
    'district':    ['Parc', 'Rocade', 'Gare', 'Parc', 'Rocade'],
    'floor_area':  [60, 60, 60, 45, 80],
    'distance': [2.0, 2.0, 2.0, 1.0, 3.0],
    'price':   [223, 200, 210, 186, 248],
})

# D2.5 — the five two-room apartments (T2) in the South with their distance to the center (lesson 2.5)
t2_south = pd.DataFrame({
    'sale':       ['S03', 'S04', 'S05', 'S06', 'S07'],
    'floor_area':  [43, 40, 44, 46, 52],
    'distance': [3.5, 2.0, 1.5, 2.5, 0.5],
})

# D2.E2 — three homes put up for sale (exercise 2.2)
new_homes = pd.DataFrame({
    'home':    ['H', 'I', 'J'],
    'floor_area':  [40, 75, 90],
    'distance': [0.5, 3.5, 1.0],
})

### Data variables

Code names are in English. For the columns, the last column gives the name used in the book's CSV files.

| Name | Meaning | Unit | CSV column |
|:--|:--|:--|:--|
| `home` | home identifier (A, B…) | — | `logement` |
| `floor_area` | living area of the home | m² | `surface_m2` |
| `price` | sale price | k€ | `prix_kEUR` |
| `distance` | distance to the city center | km | `distance_km` |
| `district` | district of the home | — | `quartier` |
| `sale` | sale identifier (S01…) | — | `vente` |

## The lesson computations, in Python

These cells redo the computations of lessons 2.1 to 2.5. Run them and compare with the book.

In [ ]:
# Lesson 2.1 — the agency's rule applied to G, term by term (equation 2.1)
area_part = 3 * 62        # k€/m² × m² = k€
distance_part = -8 * 2.5     # k€/km × km = k€
fixed_part = 45               # k€
price_G = area_part + distance_part + fixed_part
print(area_part, distance_part, fixed_part, '→', price_G, 'k€; signed deviation:', 209 - price_G, 'k€')

In [ ]:
# Lesson 2.2 — vectors, dot product (equation 2.2) and prediction (equation 2.3)
x_G = np.array([62, 2.5])    # (floor area in m², distance in km)
w = np.array([3, -8])        # (k€/m², k€/km), same order as x
b = 45                       # k€
print('dimension of x_G:', x_G.shape, '  w · x_G =', np.dot(w, x_G), '  ŷ =', np.dot(w, x_G) + b)
print('weights in the wrong order:', np.dot(w[::-1], x_G) + b, 'k€ (absurd)')

In [ ]:
# Lesson 2.3 — all predictions at once (equation 2.5)
X = homes[['floor_area', 'distance']].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)
y_hat = X @ w + b
print('X :', X.shape, ' w :', w.shape, ' ŷ :', y_hat.shape)
print('ŷ =', y_hat)
print('signed deviations y − ŷ =', y - y_hat)
print('mean absolute deviation, 7 homes:', round(np.mean(np.abs(y - y_hat)), 2), 'k€')
print('same, A to F:', round(np.mean(np.abs(y - y_hat)[:6]), 2), 'k€; line from the chapter opening, A to F:',
      round(np.mean(np.abs(y - (3 * X[:, 0] + 30))[:6]), 2), 'k€')

In [ ]:
# Lesson 2.4 — indicator columns (equation 2.6), with Gare (the station) as the reference
dummies = pd.get_dummies(east['district'], drop_first=True).astype(int)   # columns Parc, Rocade
X_east = np.column_stack([east['floor_area'], east['distance'], dummies['Parc'], dummies['Rocade']])
w_east = np.array([3, -8, 12, -10])
y_east = east['price'].to_numpy(dtype=float)
print(dummies.columns.tolist(), ' X:', X_east.shape)
print('with the district:', X_east @ w_east + 45)
print('without the district:', X_east[:, :2] @ w_east[:2] + 45)
print('mean absolute deviation: without', np.mean(np.abs(y_east - (X_east[:, :2] @ w_east[:2] + 45))),
      ' with', np.mean(np.abs(y_east - (X_east @ w_east + 45))))

In [ ]:
# Lesson 2.5 — the same prediction on two scales (equation 2.7), division by n
X5 = t2_south[['floor_area', 'distance']].to_numpy(dtype=float)
mu, sigma = X5.mean(axis=0), X5.std(axis=0, ddof=0)
w, b = np.array([3, -8]), 45
w_std, b_std = w * sigma, b + np.sum(w * mu)
Z = (X5 - mu) / sigma
print('means:', mu, ' standard deviations:', sigma)
print('standardized weights:', w_std, ' intercept:', b_std)
print('original units:', X5 @ w + b)
print('standardized scale:', Z @ w_std + b_std)

## Exercise 2.1 — Reading a diagram

*Concepts: product of a matrix and a vector (lesson 2.3).* Go back to the matrix diagram of lesson 2.3 (the seven homes A to G). The cell below rebuilds $X$, $\mathbf{w}$ and $b$ to check your answers.

In [ ]:
X = homes[['floor_area', 'distance']].to_numpy(dtype=float)
w, b = np.array([3, -8]), 45
print(X)

In [ ]:
# TO COMPLETE: your answers, read from the diagram
dimensions = {'X': None, 'w': None, 'y_hat': None}   # e.g. (7, 2), (2,), (7,)
row_280 = None             # letter of the home whose prediction is 280 k€
dimensions_c = {'X': None, 'w': None, 'y_hat': None}   # with the floor level and an 8th home

In [ ]:
# Check
y_hat = X @ w + b
truth = {'X': X.shape, 'w': w.shape, 'y_hat': y_hat.shape}
if None in dimensions.values() or row_280 is None or None in dimensions_c.values():
    print('Complete the previous cell.')
else:
    print('a) dimensions:', 'correct' if dimensions == truth else 'check again')
    letter = homes['home'][int(np.argmin(np.abs(y_hat - 280)))]
    print('b) home:', 'correct' if row_280 == letter else 'check again')
    X8 = np.zeros((8, 3)); w3 = np.zeros(3)   # shape only: 8 homes, 3 features
    print('c) dimensions:', 'correct' if dimensions_c == {'X': X8.shape, 'w': w3.shape,
          'y_hat': (X8 @ w3).shape} else 'check again')

*Your explanations:* …

## Exercise 2.2 — Three new homes

*Data D2.E2. Concepts: dot product (lesson 2.2), product of a matrix and a vector (lesson 2.3).* Do questions a) to c) on paper first, with the table of exercise 2.2 in the book.

In [ ]:
new_homes

### a) and b) On paper

Write the three vectors, the matrix $X$ and its dimensions, then the three predicted prices with their units. Enter your prices below.

In [ ]:
# TO COMPLETE: your predicted prices computed by hand, in k€
price_by_hand = {'H': None, 'I': None, 'J': None}

In [ ]:
# Check (independent computation, row by row)
for _, l in new_homes.iterrows():
    truth = 3 * l['floor_area'] - 8 * l['distance'] + 45
    yours = price_by_hand[l['home']]
    status = 'not yet completed' if yours is None else ('correct' if np.isclose(yours, truth) else 'check again')
    print(f"{l['home']} : {status}")

### c) Interpret

*Your answer:* …

### d) All predictions in one line

Complete the cell, then run the check.

In [ ]:
# X: one row per home (H, I, J),
#    columns floor area then distance
X = None          # TO COMPLETE
w = None          # TO COMPLETE: order of the columns of X
b = None          # TO COMPLETE: in k€
y_hat = None  # TO COMPLETE: equation 2.5

<details><summary><b>Hint</b> (click if you get stuck)</summary>

`new_homes[['floor_area', 'distance']].to_numpy()` gives the matrix. `np.array([..., ...])` creates a vector. The matrix product is written `@`.

</details>

In [ ]:
# Check
if y_hat is None:
    print('Complete the previous cell.')
else:
    print('dimensions:', np.shape(X), '@', np.shape(w), '→', np.shape(y_hat))
    print('dimensions:', 'correct' if np.shape(X) == (3, 2) and np.shape(y_hat) == (3,) else 'check again')
    print('predictions:', y_hat, '→',
          'correct' if np.allclose(y_hat, [161, 242, 307]) else 'check again')

### e) Predict, then check

Columns of $X$ in the order distance then floor area, $\mathbf{w}$ unchanged. First write down your prediction of the sign of the three prices, then run the cell.

*Your prediction:* …

In [ ]:
if y_hat is not None:
    X_inverse = X[:, ::-1]          # columns swapped: distance, floor area
    print('columns swapped:', X_inverse @ w + b)

## Exercise 2.3 — What do the weights say?

*Concepts: reading a weight (lesson 2.2), indicator columns (lesson 2.4).* Rule: $\mathbf{w} = (3, -8, 12, -10)$ for floor area, distance, Parc, Rocade; $b = 45$ k€; Gare is the reference. Questions on paper, check below.

In [ ]:
# TO COMPLETE: differences in predicted price, in k€ (first home − second)
a_park_minus_ring_road = None
b_far_minus_near = None   # 1.5 km farther, same district, same floor area
c_far_park_minus_station = None     # Parc and 1.5 km farther, compared with Gare

In [ ]:
# Check: we compute the rule for pairs of fictitious homes
rule = lambda floor_area, distance, park, ring_road: 3 * floor_area - 8 * distance + 12 * park - 10 * ring_road + 45
truths = {'a': rule(60, 2, 1, 0) - rule(60, 2, 0, 1),
         'b': rule(60, 3.5, 0, 0) - rule(60, 2, 0, 0),
         'c': rule(60, 3.5, 1, 0) - rule(60, 2, 0, 0)}
for k, yours in [('a', a_park_minus_ring_road), ('b', b_far_minus_near), ('c', c_far_park_minus_station)]:
    status = 'not yet completed' if yours is None else ('correct' if np.isclose(yours, truths[k]) else 'check again')
    print(f'{k}) {status}')

*Your interpretations for c) and d):* …

## Exercise 2.4 — Indicator columns in Python

*Data D2.4. Concepts: indicator columns (lesson 2.4), product of a matrix and a vector (lesson 2.3).*

In [ ]:
east

In [ ]:
# TO COMPLETE
park = None     # a) 1 if the home is in Parc, 0 otherwise (integers)
ring_road = None   # a) 1 if the home is in Rocade, 0 otherwise
X4 = None       # b) 5 × 4: floor area, distance, Parc, Rocade
w4 = np.array([3, -8, 12, -10])
b4 = 45
y4 = None       # b) equation 2.5

<details><summary><b>Hint</b> (click if you get stuck)</summary>

`(east['district'] == 'Parc')` gives true/false values; `.astype(int)` turns them into 1/0. `np.column_stack([a, b, c, d])` places four columns side by side.

</details>

In [ ]:
# Check
if y4 is None:
    print('Complete the previous cell.')
else:
    print('indicator columns:', 'correct' if list(park) == [1, 0, 0, 1, 0] and list(ring_road) == [0, 1, 0, 0, 1] else 'check again')
    print('X4 :', np.shape(X4), '→', 'correct' if np.shape(X4) == (5, 4) else 'check again')
    print('predictions:', y4, '→', 'correct' if np.allclose(y4, [221, 199, 209, 184, 251]) else 'check again')

**c)** Mean absolute deviation, with and without the district columns.

In [ ]:
# TO COMPLETE
y_obs = east['price'].to_numpy(dtype=float)
mae_without = None   # without the district: first two columns and first two weights
mae_with = None

In [ ]:
if mae_without is None or mae_with is None:
    print('Complete the previous cell.')
else:
    print(f'without: {mae_without:.1f} k€, with: {mae_with:.1f} k€ →',
          'correct' if np.isclose(mae_without, 10.2) and np.isclose(mae_with, 1.8) else 'check again')

*Your comment:* …

## Exercise 2.5 — Predict, then check: changing units

*Data D2.5. Concepts: change of scale (lesson 2.5, equation 2.7).* Write down your predictions for a) and b) before running the cells.

*Your predictions:* …

In [ ]:
# TO COMPLETE: conversion to standardized weights (equation 2.7)
def standardized_weights(w, b, mu, sigma):
    w_std = None   # TO COMPLETE
    b_std = None   # TO COMPLETE
    return w_std, b_std

In [ ]:
# Two versions of the same rule: distance in km, then in meters
X_km = t2_south[['floor_area', 'distance']].to_numpy(dtype=float)
X_m = X_km * np.array([1, 1000])
w_km, w_m, b = np.array([3, -8]), np.array([3, -0.008]), 45
print('predictions (km):', X_km @ w_km + b)
print('predictions (m): ', X_m @ w_m + b)
results = {}
for name, Xv, wv in [('km', X_km, w_km), ('m', X_m, w_m)]:
    mu, sigma = Xv.mean(axis=0), Xv.std(axis=0, ddof=0)
    results[name] = standardized_weights(wv, b, mu, sigma)
    print(name, ': mean distance =', mu[1], ', standard deviation =', sigma[1], ', standardized weights =', results[name])

In [ ]:
# Check
if results['km'][0] is None:
    print('Complete the function standardized_weights.')
else:
    ok = all(np.allclose(results[k][0], [12, -8]) and np.isclose(results[k][1], 164) for k in results)
    print('standardized weights identical in km and in m:', 'correct' if ok else 'check again')

In [ ]:
if results['km'][0] is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 2.6))
    for ax, title, values in [
            (axes[0], 'Raw weight of the 2nd term', {'km': w_km[1], 'm': w_m[1]}),
            (axes[1], 'Standardized weight of the 2nd term', {k: results[k][0][1] for k in results})]:
        ax.bar(list(values), list(values.values()), color=[BLUE, ORANGE])
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()

*Your conclusion for c):* …

## Exercise 2.6 — Challenge: one rule, several ways to write it

*Data D2.4. Concepts: lessons 2.3, 2.4 and 2.5.* Three indicator columns: Parc, Rocade, Gare.

In [ ]:
station = (east['district'] == 'Gare').astype(int).to_numpy()
park3 = (east['district'] == 'Parc').astype(int).to_numpy()
ring_road3 = (east['district'] == 'Rocade').astype(int).to_numpy()
X3 = np.column_stack([east['floor_area'], east['distance'], park3, ring_road3, station])
print('sum of the three indicators, row by row:', X3[:, 2:].sum(axis=1))
print('the colleague’s rule (b = 40):', X3 @ np.array([3, -8, 17, -5, 5]) + 40)

In [ ]:
# TO COMPLETE b): three district weights (Parc, Rocade, Gare) for b = 50
district_weight_b50 = None   # list of three numbers

In [ ]:
# Check
if district_weight_b50 is None:
    print('Complete the previous cell.')
else:
    y50 = X3 @ np.array([3, -8] + list(district_weight_b50)) + 50
    print('predictions with b = 50:', y50, '→', 'correct' if np.allclose(y50, [221, 199, 209, 184, 251]) else 'check again')

*Your answer to c):* …